In [ ]:
!pip install earthengine-api --upgrade
!pip install folium
!pip install pyproj

In [ ]:
import ee
import folium
ee.Authenticate()
ee.Initialize(project='ee-kulchiranut')

Chack Photo

In [ ]:

roi = ee.Geometry.Polygon([
    [[102.49856452615988, 17.725056403180492],
     [102.91329841287863, 17.725056403180492],
     [102.91329841287863, 18.047864078632685],
     [102.49856452615988, 18.047864078632685],
     [102.49856452615988, 17.725056403180492]]
])

def check_availability(collection_id, date_start, date_end, label=""):  #เหตุผลที่ใส่ label = เเยกภาพ 3 ดาวเทียม
    col = ee.ImageCollection(collection_id) \
        .filterBounds(roi) \
        .filterDate(date_start, date_end) \
        .sort('CLOUD_COVER')
    n = col.size().getInfo()
    print(f"\n📅 {label} | {collection_id}")
    print(f"   ช่วง {date_start} ถึง {date_end} (ไม่รวมวันสุดท้าย) → พบ {n} ภาพ")
    if n > 0:
        names  = col.aggregate_array('LANDSAT_PRODUCT_ID').getInfo()
        dates  = col.aggregate_array('DATE_ACQUIRED').getInfo()
        clouds = col.aggregate_array('CLOUD_COVER').getInfo()
        for nm, d, c in zip(names, dates, clouds):
            print(f"   🛰 {nm}")
            print(f"      ถ่ายวันที่ {d}   เมฆ {c:.1f}%")
        print(f"   ✅ ภาพที่ .first() จะเลือกใช้ = {dates[0]} (เมฆน้อยสุด {clouds[0]:.1f}%)")
    else:
        print("   ❌ ไม่มีภาพเลย → ลองขยายช่วงวันที่ หรือเปลี่ยนดาวเทียม")
    return n
                                                                            #คำเเทนที่ label=''
check_availability("LANDSAT/LT05/C02/T1_L2", '1993-01-01', '1993-12-31', "หน้าแล้ง 1993 (Landsat 5)")
check_availability("LANDSAT/LE07/C02/T1_L2", '2003-01-01', '2003-12-31', "หน้าแล้ง 2003 (Landsat 7)")
check_availability("LANDSAT/LC09/C02/T1_L2", '2026-01-01', '2026-12-31', "หน้าแล้ง 2026 (Landsat 9)")

Map 1 การทำ NDWI เเละหาความเปลี่ยนเเปลง หากจะรัน Map 2 รันโค้ดนี้ด้วย เพราะมีภาพเชื่อมโยงกัน

In [ ]:
import ee
import folium
import geopandas as gpd  # สำหรับอ่านไฟล์ Shapefile ท้องถิ่น
import json              # สำหรับแปลงข้อมูลเป็นรูปแบบที่ GEE เข้าใจ

#ROI
lat_min, lon_min = 17.880568, 102.722825
lat_max, lon_max = 17.875089, 102.722825

roi = ee.Geometry.Polygon([
    [[102.49856452615988, 17.725056403180492],
     [102.91329841287863, 17.725056403180492],
     [102.91329841287863, 18.047864078632685],
     [102.49856452615988, 18.047864078632685],
     [102.49856452615988, 17.725056403180492]]
])


DRY_1993 = ('1993-01-01', '1993-11-30')
DRY_2003 = ('2003-01-01', '2003-04-30')
DRY_2026 = ('2026-01-01', '2026-06-01')

#SHP
SHAPEFILE_PATH = '/content/ขอบเขตระยะ2กิโลเมตร.shp'

try:
    gdf = gpd.read_file(SHAPEFILE_PATH)
    gdf = gdf.to_crs(epsg=4326)
    geojson_dict = json.loads(gdf.to_json())
    shapefile_fc = ee.FeatureCollection(geojson_dict)
    clip_geometry = shapefile_fc.geometry()
    print("👍 โหลดข้อมูล Shapefile จากพาร์ทท้องถิ่นสำเร็จ! ระบบจะทำการ Clip และแปลงข้อมูลตามพื้นที่นี้")
except Exception as e:
    shapefile_fc = ee.FeatureCollection(roi)
    clip_geometry = roi
    print(f"⚠️ คำเตือน: ไม่สามารถโหลดไฟล์ Shapefile ได้เนื่องจาก: {e}")
    print("ระบบจะหันไปใช้ขอบเขตพื้นที่ศึกษา (ROI) สี่เหลี่ยมผืนผ้าเดิมทำงานแทนชั่วคราว")

Map = folium.Map(location=[(lat_min + lat_max) / 2, (lon_min + lon_max) / 2], zoom_start=9)

# แผ่นที่ฐาน (Base Map)
folium.TileLayer(
    tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
    attr='Google Satellite',
    name='Google Satellite',
    overlay=False,
    control=True
).add_to(Map)

vis_roi = {'color': 'FFFF00', 'fillColor': '00000000', 'width': 2}
vis_shape_edge = {'color': 'FF00FF', 'fillColor': '00000000', 'width': 3}
ndwi_palette = ['#a52a2a', '#ffffff', '#0000ff']

# สีเขียว = ทับถม (Deposition) / สีขาว = ไม่เปลี่ยนแปลง (No Change) / สีแดง = กัดเซาะ (Erosion)
class3_palette = ['#00A000', '#FFFFFF', '#FF0000']

# Landsat 5 และ Landsat 7 เป็นตระกูล TM/ETM+ เหมือนกัน → เรียงแบนด์สีจริงชุดเดียวกัน
vis_params_l5 = {'bands': ['SR_B3', 'SR_B2', 'SR_B1'], 'min': 0.0, 'max': 0.3}
vis_params_l7 = {'bands': ['SR_B3', 'SR_B2', 'SR_B1'], 'min': 0.0, 'max': 0.3}
vis_params_l9 = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0.0, 'max': 0.3}
vis_ndwi = {'min': -1, 'max': 1, 'palette': ndwi_palette}

# แสดงผลค่าการเปลี่ยนแปลงแบบ Float เต็มช่วง -1 ถึง 1
vis_change_float = {'min': -1, 'max': 1, 'palette': class3_palette}

# แสดงผลภาพจัดหมวดหมู่ 3 คลาส (1 = ทับถม, 2 = ไม่เปลี่ยนแปลง, 3 = กัดเซาะ)
vis_class3 = {'min': 1, 'max': 3, 'palette': class3_palette}

# เกณฑ์ตัดสินการเปลี่ยนแปลง (ปรับค่าได้ตามความเหมาะสมของพื้นที่ศึกษา)
CHANGE_THRESHOLD = 0.1

#เช็คภาพ
def print_image_info(image, n, label):
    name  = image.get('LANDSAT_PRODUCT_ID').getInfo()
    date  = image.get('DATE_ACQUIRED').getInfo()
    cloud = image.get('CLOUD_COVER').getInfo()
    print(f"🛰 {label}")
    print(f"   ชื่อภาพ  : {name}")
    print(f"   ถ่ายวันที่: {date}   เมฆ {cloud:.1f}%   (คัดจากทั้งหมด {n} ภาพในช่วงหน้าแล้ง)")

def process_landsat5(date_start, date_end, label='Landsat 5'):
    collection = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2") \
        .filterBounds(roi) \
        .filterDate(date_start, date_end) \
        .sort('CLOUD_COVER')

    n = collection.size().getInfo()
    if n == 0:
        raise ValueError(f"❌ {label}: ไม่พบภาพช่วง {date_start} ถึง {date_end} → ลองขยายช่วงวันที่")

    image = ee.Image(collection.first())
    print_image_info(image, n, label)

    def apply_scale_factors(img):
        optical_bands = img.select('SR_B.').multiply(0.0000275).add(-0.2)
        thermal_bands = img.select('ST_B6').multiply(0.00341802).add(149.0)
        return img.addBands(optical_bands, None, True).addBands(thermal_bands, None, True)

    image = apply_scale_factors(image)
    ndwi = image.normalizedDifference(['SR_B2', 'SR_B4']).rename('NDWI').toFloat().clamp(-1, 1)
    return image, ndwi

def process_landsat7(date_start, date_end, label='Landsat 7'):
    collection = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2") \
        .filterBounds(roi) \
        .filterDate(date_start, date_end) \
        .sort('CLOUD_COVER')

    n = collection.size().getInfo()
    if n == 0:
        raise ValueError(f"❌ {label}: ไม่พบภาพช่วง {date_start} ถึง {date_end} → ลองขยายช่วงวันที่")

    image = ee.Image(collection.first())
    print_image_info(image, n, label)

    def apply_scale_factors(img):
        optical_bands = img.select('SR_B.').multiply(0.0000275).add(-0.2)
        thermal_bands = img.select('ST_B6').multiply(0.00341802).add(149.0)
        return img.addBands(optical_bands, None, True).addBands(thermal_bands, None, True)

    image = apply_scale_factors(image)
    ndwi = image.normalizedDifference(['SR_B2', 'SR_B4']).rename('NDWI').toFloat().clamp(-1, 1)
    return image, ndwi

def process_landsat9(date_start, date_end, label='Landsat 9'):
    collection = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2") \
        .filterBounds(roi) \
        .filterDate(date_start, date_end) \
        .sort('CLOUD_COVER')

    n = collection.size().getInfo()
    if n == 0:
        raise ValueError(f"❌ {label}: ไม่พบภาพช่วง {date_start} ถึง {date_end} → ลองขยายช่วงวันที่")

    image = ee.Image(collection.first())
    print_image_info(image, n, label)

    def apply_scale_factors(img):
        optical_bands = img.select('SR_B.').multiply(0.0000275).add(-0.2)
        thermal_bands = img.select('ST_B10').multiply(0.00341802).add(149.0)
        return img.addBands(optical_bands, None, True).addBands(thermal_bands, None, True)

    image = apply_scale_factors(image)
    ndwi = image.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI').toFloat().clamp(-1, 1)
    return image, ndwi

# ==============================================================================
img_1993, ndwi_1993 = process_landsat5(DRY_1993[0], DRY_1993[1], 'หน้าแล้ง 1993 (Landsat 5)')
img_2003, ndwi_2003 = process_landsat7(DRY_2003[0], DRY_2003[1], 'หน้าแล้ง 2003 (Landsat 7)')
img_2026, ndwi_2026 = process_landsat9(DRY_2026[0], DRY_2026[1], 'หน้าแล้ง 2026 (Landsat 9)')

# ผลต่าง NDWI แบบ Float และ Clamp ให้อยู่ในช่วง -1 ถึง 1
# ค่าบวก (NDWI เพิ่ม) = พื้นดินกลายเป็นน้ำ = การกัดเซาะ (Erosion)
# ค่าลบ (NDWI ลด)   = น้ำกลายเป็นพื้นดิน = การทับถม (Deposition)
change_93_03 = ndwi_2003.subtract(ndwi_1993).toFloat().clamp(-1, 1)
change_03_26 = ndwi_2026.subtract(ndwi_2003).toFloat().clamp(-1, 1)
change_93_26 = ndwi_2026.subtract(ndwi_1993).toFloat().clamp(-1, 1)

change_93_03_clipped = change_93_03.clip(clip_geometry)
change_03_26_clipped = change_03_26.clip(clip_geometry)
change_93_26_clipped = change_93_26.clip(clip_geometry)

# ------------------------------------------------------------------------------
# ฟังก์ชันจัดหมวดหมู่เพียง 3 ประเภท และแปลงเป็น Vector
#   คลาส 1 = การทับถม (Deposition)      : ผลต่าง NDWI <= -CHANGE_THRESHOLD
#   คลาส 2 = ไม่เปลี่ยนแปลง (No Change)  : -CHANGE_THRESHOLD < ผลต่าง < CHANGE_THRESHOLD
#   คลาส 3 = การกัดเซาะ (Erosion)       : ผลต่าง NDWI >= CHANGE_THRESHOLD
# ------------------------------------------------------------------------------
def reclassify_3class(change_image):
    change_float = change_image.toFloat().clamp(-1, 1)
    classified = ee.Image(2) \
        .where(change_float.lte(-CHANGE_THRESHOLD), 1) \
        .where(change_float.gte(CHANGE_THRESHOLD), 3) \
        .rename('class')
    return classified.clip(clip_geometry)

def vectorize_3class(classified_image):
    vectors = classified_image.reduceToVectors(
        geometry=clip_geometry,
        crs='EPSG:32648',
        scale=30,
        geometryType='polygon',
        eightConnected=True,
        labelProperty='zone_id',
        maxPixels=1e9
    )
    return vectors

class_93_03 = reclassify_3class(change_93_03_clipped)
class_03_26 = reclassify_3class(change_03_26_clipped)
class_93_26 = reclassify_3class(change_93_26_clipped)

vector_93_03 = vectorize_3class(class_93_03)
vector_03_26 = vectorize_3class(class_03_26)
vector_93_26 = vectorize_3class(class_93_26)


# --- กลุ่มภาพสีจริง (True Color) ---
folium.TileLayer(
    tiles=img_1993.getMapId(vis_params_l5)['tile_fetcher'].url_format,
    attr='GEE - Landsat 5', name='หน้าแล้ง 1993 (True Color, L5)', overlay=True, control=True, show=False
).add_to(Map)

folium.TileLayer(
    tiles=img_2003.getMapId(vis_params_l7)['tile_fetcher'].url_format,
    attr='GEE - Landsat 7', name='หน้าแล้ง 2003 (True Color, L7)', overlay=True, control=True, show=False
).add_to(Map)

folium.TileLayer(
    tiles=img_2026.getMapId(vis_params_l9)['tile_fetcher'].url_format,
    attr='GEE - Landsat 9', name='หน้าแล้ง 2026 (True Color, L9)', overlay=True, control=True, show=False
).add_to(Map)

# --- กลุ่มการวิเคราะห์ NDWI (Float, -1 ถึง 1) ---
folium.TileLayer(
    tiles=ndwi_1993.getMapId(vis_ndwi)['tile_fetcher'].url_format,
    attr='GEE - NDWI', name='หน้าแล้ง 1993 (NDWI Float)', overlay=True, control=True, show=False
).add_to(Map)

folium.TileLayer(
    tiles=ndwi_2003.getMapId(vis_ndwi)['tile_fetcher'].url_format,
    attr='GEE - NDWI', name='หน้าแล้ง 2003 (NDWI Float)', overlay=True, control=True, show=False
).add_to(Map)

folium.TileLayer(
    tiles=ndwi_2026.getMapId(vis_ndwi)['tile_fetcher'].url_format,
    attr='GEE - NDWI', name='หน้าแล้ง 2026 (NDWI Float)', overlay=True, control=True, show=False
).add_to(Map)

# --- กลุ่มค่าการเปลี่ยนแปลงแบบ Float (ต่อเนื่อง -1 ถึง 1) ---
folium.TileLayer(
    tiles=change_93_03_clipped.getMapId(vis_change_float)['tile_fetcher'].url_format,
    attr='GEE - Change Float', name='ผลต่าง NDWI Float (1993-2003)', overlay=True, control=True, show=False
).add_to(Map)

folium.TileLayer(
    tiles=change_03_26_clipped.getMapId(vis_change_float)['tile_fetcher'].url_format,
    attr='GEE - Change Float', name='ผลต่าง NDWI Float (2003-2026)', overlay=True, control=True, show=False
).add_to(Map)

folium.TileLayer(
    tiles=change_93_26_clipped.getMapId(vis_change_float)['tile_fetcher'].url_format,
    attr='GEE - Change Float', name='ผลต่าง NDWI Float (1993-2026)', overlay=True, control=True, show=False
).add_to(Map)

# --- กลุ่มผลจัดหมวดหมู่ 3 ประเภท (เขียว = ทับถม / ขาว = ไม่เปลี่ยนแปลง / แดง = กัดเซาะ) ---
folium.TileLayer(
    tiles=class_93_03.getMapId(vis_class3)['tile_fetcher'].url_format,
    attr='GEE - 3 Class', name='3 ประเภท กัดเซาะ/ทับถม (1993-2003)', overlay=True, control=True, show=True
).add_to(Map)

folium.TileLayer(
    tiles=class_03_26.getMapId(vis_class3)['tile_fetcher'].url_format,
    attr='GEE - 3 Class', name='3 ประเภท กัดเซาะ/ทับถม (2003-2026)', overlay=True, control=True, show=True
).add_to(Map)

folium.TileLayer(
    tiles=class_93_26.getMapId(vis_class3)['tile_fetcher'].url_format,
    attr='GEE - 3 Class', name='3 ประเภท กัดเซาะ/ทับถมรวม (1993-2026)', overlay=True, control=True, show=True
).add_to(Map)

# --- ขอบเขตพื้นที่ศึกษา (ROI เดิม) ---
Map.add_child(folium.raster_layers.TileLayer(
    tiles=ee.Image().paint(roi, 0, 3).getMapId(vis_roi)['tile_fetcher'].url_format,
    attr='Google Earth Engine', name='ROI', overlay=True, control=True
))

# แสดงกล่องควบคุมเลเยอร์
folium.LayerControl(position='topright', collapsed=False).add_to(Map)

Map

กรณี เเยกน้ำ เเละ ตลิ่ง

In [ ]:
# ==============================================================================
# เกณฑ์แยกน้ำ/ดิน แบบกำหนด "เฉพาะปี" ได้อิสระ
# เหตุผล: ความขุ่นตะกอนของน้ำโขงแต่ละปีไม่เท่ากัน เกณฑ์เดียวจึงอาจไม่พอดีทุกปี
# ==============================================================================
THRESHOLD_1993 = -0.15   # น้ำขุ่นมาก ต้องกดเกณฑ์ลงต่ำ
THRESHOLD_2003 = 0.0     # ใช้ค่าตามนิยามตำราไปก่อน แล้วดูผลด้วยตา
THRESHOLD_2026 = 0.0

def reclassify_bank_river(ndwi_image, threshold):
    """คืนภาพที่พิกเซลตลิ่ง = 1 ส่วนพิกเซลแม่น้ำถูก Mask ทิ้งเป็นค่าว่างเปล่า
    threshold: เกณฑ์แยกน้ำ/ดินของภาพใบนั้น (NDWI < threshold = ตลิ่ง)"""
    ndwi_c = ndwi_image.clip(clip_geometry)

    # 1. แบ่ง 2 ประเภท: NDWI < เกณฑ์ = ตลิ่ง (ได้ 1) / >= เกณฑ์ = แม่น้ำ (ได้ 0)
    land_mask = ndwi_c.lt(threshold)

    # 2. selfMask() = ตัดพิกเซลค่า 0 (แม่น้ำ) ออกเป็นค่าว่าง เหลือเฉพาะพิกเซลค่า 1 (ตลิ่ง)
    bank_only = land_mask.selfMask()

    # 3. reduceToVectors รับเฉพาะแบนด์จำนวนเต็ม จึงบังคับ toInt()
    return bank_only.rename('bank').toInt()


bank_1993 = reclassify_bank_river(ndwi_1993, THRESHOLD_1993)
bank_2003 = reclassify_bank_river(ndwi_2003, THRESHOLD_2003)
bank_2026 = reclassify_bank_river(ndwi_2026, THRESHOLD_2026)
# ==============================================================================
# สร้างแผนที่ใบใหม่ (Map_bank) — ไม่ยุ่งกับ Map เดิมจาก STEP 5 เลย
# จึงมี LayerControl ตัวเดียวแน่นอน กล่องควบคุมแสดงผลปกติ
# ==============================================================================
Map_bank = folium.Map(location=[(lat_min + lat_max) / 2, (lon_min + lon_max) / 2], zoom_start=11)

# แผนที่ฐาน: ภาพถ่ายดาวเทียม Google (ไว้เทียบแนวตลิ่งจริงกับผลการ Mask)
folium.TileLayer(
    tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
    attr='Google Satellite',
    name='Google Satellite',
    overlay=False,
    control=True
).add_to(Map_bank)

# --- ภาพสีจริงของแต่ละปี (เปิดเทียบกับผลตลิ่งปีเดียวกันเพื่อตรวจเกณฑ์ด้วยตา) ---
folium.TileLayer(
    tiles=img_1993.getMapId(vis_params_l5)['tile_fetcher'].url_format,
    attr='GEE - Landsat 5', name='หน้าแล้ง 1993 (True Color, L5)', overlay=True, control=True, show=False
).add_to(Map_bank)

folium.TileLayer(
    tiles=img_2003.getMapId(vis_params_l7)['tile_fetcher'].url_format,
    attr='GEE - Landsat 7', name='หน้าแล้ง 2003 (True Color, L7)', overlay=True, control=True, show=False
).add_to(Map_bank)

folium.TileLayer(
    tiles=img_2026.getMapId(vis_params_l9)['tile_fetcher'].url_format,
    attr='GEE - Landsat 9', name='หน้าแล้ง 2026 (True Color, L9)', overlay=True, control=True, show=False
).add_to(Map_bank)

# --- ผลพื้นที่ตลิ่ง (แม่น้ำเป็นช่องว่าง มองทะลุลงไปเห็นแผนที่ฐาน) ---
vis_bank = {'min': 1, 'max': 1, 'palette': ['#D2B48C']}  # ตลิ่งสีน้ำตาลอ่อน

folium.TileLayer(
    tiles=bank_1993.getMapId(vis_bank)['tile_fetcher'].url_format,
    attr='GEE - Bank',
    name=f'ตลิ่ง 1993 (เกณฑ์ {THRESHOLD_1993})',
    overlay=True, control=True, show=True
).add_to(Map_bank)

folium.TileLayer(
    tiles=bank_2003.getMapId(vis_bank)['tile_fetcher'].url_format,
    attr='GEE - Bank',
    name=f'ตลิ่ง 2003 (เกณฑ์ {THRESHOLD_2003})',
    overlay=True, control=True, show=True
).add_to(Map_bank)

folium.TileLayer(
    tiles=bank_2026.getMapId(vis_bank)['tile_fetcher'].url_format,
    attr='GEE - Bank',
    name=f'ตลิ่ง 2026 (เกณฑ์ {THRESHOLD_2026})',
    overlay=True, control=True, show=True
).add_to(Map_bank)

Map_bank.add_child(folium.raster_layers.TileLayer(
    tiles=ee.Image().paint(roi, 0, 3).getMapId(vis_roi)['tile_fetcher'].url_format,
    attr='Google Earth Engine', name='ROI', overlay=True, control=True
))

# LayerControl ตัวเดียว เพิ่มเป็นลำดับสุดท้ายเสมอ
folium.LayerControl(position='topright', collapsed=False).add_to(Map_bank)

print(f"🔎 แผนที่ตรวจตลิ่ง | เกณฑ์: 1993={THRESHOLD_1993}, 2003={THRESHOLD_2003}, 2026={THRESHOLD_2026}")
Map_bank

Raster to Vector ตัดเเม่น้ำ (Map2)

In [ ]:
# ==============================================================================
# แปลงพื้นที่ตลิ่ง Raster → Vector และส่งออก SHP ลงคอมพิวเตอร์ ทั้ง 3 ปี
# ==============================================================================
import urllib.request
from google.colab import files
import time

def vectorize_bank(bank_image):
    """แปลงพื้นที่ตลิ่งเป็น Polygon — พิกเซลแม่น้ำที่ถูก Mask จะถูกข้ามอัตโนมัติ
    Polygon ที่ได้จึงเป็นแผ่นตลิ่งที่มีช่องว่างตรงลำน้ำพอดี"""
    return bank_image.reduceToVectors(
        geometry=clip_geometry,
        crs='EPSG:32648',
        scale=30,
        geometryType='polygon',
        eightConnected=True,
        labelProperty='bank_id',
        maxPixels=1e9
    )

bank_vector_1993 = vectorize_bank(bank_1993)
bank_vector_2003 = vectorize_bank(bank_2003)
bank_vector_2026 = vectorize_bank(bank_2026)

def direct_download_bank_shp(feature_collection, filename):
    try:
        print(f"⏳ กำลังขอลิงก์สร้างไฟล์ SHP จากเซิร์ฟเวอร์ GEE สำหรับ: {filename}...")
        download_url = feature_collection.getDownloadURL(filetype='SHP', filename=filename)
        local_zip_path = f"/content/{filename}.zip"
        urllib.request.urlretrieve(download_url, local_zip_path)
        print(f"📦 เริ่มการดาวน์โหลดไฟล์ {filename}.zip ลงเครื่องคอมพิวเตอร์ของคุณแล้ว...")
        files.download(local_zip_path)
        time.sleep(3)
    except Exception as e:
        print(f"❌ เกิดข้อผิดพลาดในการดาวน์โหลด {filename}: {e}")

direct_download_bank_shp(bank_vector_1993, 'RiverBank_Only_Dry_1993_L5')
direct_download_bank_shp(bank_vector_2003, 'RiverBank_Only_Dry_2003_L7')
direct_download_bank_shp(bank_vector_2026, 'RiverBank_Only_Dry_2026_L9')

print("\n✨ ส่งออกพื้นที่ตลิ่ง (ตัดแม่น้ำออก) ครบทั้ง 3 ปีแล้ว!")

In [ ]:
# ==============================================================================
# แปลงพื้นที่ตลิ่ง Raster → Vector และส่งออก SHP ลงคอมพิวเตอร์ ทั้ง 3 ปี
# ==============================================================================
import urllib.request
from google.colab import files
import time

def vectorize_bank(bank_image):
    """แปลงพื้นที่ตลิ่งเป็น Polygon — พิกเซลแม่น้ำที่ถูก Mask จะถูกข้ามอัตโนมัติ
    Polygon ที่ได้จึงเป็นแผ่นตลิ่งที่มีช่องว่างตรงลำน้ำพอดี"""
    return bank_image.reduceToVectors(
        geometry=clip_geometry,
        crs='EPSG:32648',
        scale=30,
        geometryType='polygon',
        eightConnected=True,
        labelProperty='bank_id',
        maxPixels=1e9
    )

bank_vector_1993 = vectorize_bank(bank_1993)
bank_vector_2003 = vectorize_bank(bank_2003)
bank_vector_2026 = vectorize_bank(bank_2026)

def direct_download_bank_shp(feature_collection, filename):
    try:
        print(f"⏳ กำลังขอลิงก์สร้างไฟล์ SHP จากเซิร์ฟเวอร์ GEE สำหรับ: {filename}...")
        download_url = feature_collection.getDownloadURL(filetype='SHP', filename=filename)
        local_zip_path = f"/content/{filename}.zip"
        urllib.request.urlretrieve(download_url, local_zip_path)
        print(f"📦 เริ่มการดาวน์โหลดไฟล์ {filename}.zip ลงเครื่องคอมพิวเตอร์ของคุณแล้ว...")
        files.download(local_zip_path)
        time.sleep(3)
    except Exception as e:
        print(f"❌ เกิดข้อผิดพลาดในการดาวน์โหลด {filename}: {e}")

direct_download_bank_shp(bank_vector_1993, 'RiverBank_Only_Dry_1993_L5')
direct_download_bank_shp(bank_vector_2003, 'RiverBank_Only_Dry_2003_L7')
direct_download_bank_shp(bank_vector_2026, 'RiverBank_Only_Dry_2026_L9')

print("\n✨ ส่งออกพื้นที่ตลิ่ง (ตัดแม่น้ำออก) ครบทั้ง 3 ปีแล้ว!")

Raster To Vector (Map1)

In [ ]:
# ==============================================================================
# ส่งออกไฟล์ Vector SHP (3 ประเภท) ลงคอมพิวเตอร์
# ==============================================================================
import urllib.request
from google.colab import files
import time

def direct_download_vector_shp(feature_collection, filename):
    try:
        print(f"⏳ กำลังขอลิงก์สร้างไฟล์พิกัด SHP จากเซิร์ฟเวอร์ GEE สำหรับ: {filename}...")
        # สร้าง URL สำหรับดาวน์โหลดประเภทไฟล์ Shapefile (SHP) ตรงจาก GEE
        download_url = feature_collection.getDownloadURL(filetype='SHP', filename=filename)

        # กำหนดพาร์ทเก็บไฟล์ชั่วคราวบน Google Colab
        local_zip_path = f"/content/{filename}.zip"

        # ดึงไฟล์ Zip จาก URL ของ GEE มาพักไว้ในโฟลเดอร์ของ Colab ก่อน
        urllib.request.urlretrieve(download_url, local_zip_path)

        # ใช้คำสั่งของ Colab บังคับป็อปอัปดาวน์โหลดลงคอมพิวเตอร์ทันที
        print(f"📦 เริ่มการดาวน์โหลดไฟล์ {filename}.zip ลงเครื่องคอมพิวเตอร์ของคุณแล้ว...")
        files.download(local_zip_path)

        # หน่วงเวลาสั้นๆ 3 วินาที เพื่อให้เบราว์เซอร์จัดการคิวดาวน์โหลดไฟล์ถัดไป ไม่ให้ชนกัน
        time.sleep(3)
    except Exception as e:
        print(f"❌ เกิดข้อผิดพลาดในการดาวน์โหลด {filename}: {e}")

# สั่งรันกระบวนการดาวน์โหลดเลเยอร์เวกเตอร์ 3 ประเภท (ทับถม/ไม่เปลี่ยนแปลง/กัดเซาะ) ทั้ง 3 ชุด
direct_download_vector_shp(vector_93_03, 'Erosion_3Class_Vector_Dry_1993_2003')
direct_download_vector_shp(vector_03_26, 'Erosion_3Class_Vector_Dry_2003_2026')
direct_download_vector_shp(vector_93_26, 'Erosion_3Class_Vector_Dry_1993_2026')

print("\n✨ ดำเนินการส่งข้อมูลดาวน์โหลดเรียบร้อยแล้ว!")
print("💡 หมายเหตุ: หากไฟล์ไม่ยอมเด้งโหลด กดอนุญาต Allow multiple downloads ที่มุมขวาบนของ Chrome นะครับ")

TIFF (Map1)

In [ ]:
# ==============================================================================
# ส่งออก NDWI (Float) ทั้ง 3 ปี เป็นไฟล์ GeoTIFF ลงเครื่องคอมพิวเตอร์
# ==============================================================================
import urllib.request
from google.colab import files
import time

def direct_download_ndwi_tiff(ndwi_image, filename):
    try:
        print(f"⏳ กำลังขอลิงก์สร้างไฟล์ GeoTIFF จากเซิร์ฟเวอร์ GEE สำหรับ: {filename}...")

        # 1. Clip ให้เหลือเฉพาะแนวขอบเขต Shapefile 2 กม. ก่อนส่งออก
        #    เหตุผล: ลดขนาดไฟล์ และให้ TIFF ครอบพื้นที่เดียวกับผลวิเคราะห์อื่น ๆ
        image_to_export = ndwi_image.clip(clip_geometry)

        # 2. ขอ URL ดาวน์โหลดจาก GEE
        download_url = image_to_export.getDownloadURL({
            'region': clip_geometry,     # ขอบเขตพิกัดที่จะตัดออกมา
            'scale': 30,                 # ขนาดพิกเซล 30 เมตร ตามความละเอียด Landsat
            'crs': 'EPSG:32648',         # ล็อกพิกัด UTM Zone 48N ให้ตรงกับไฟล์ Vector
            'format': 'GEO_TIFF'         # ได้เป็นไฟล์ .tif เดี่ยว ๆ
        })

        # 3. ดึงไฟล์มาพักใน Colab ก่อน
        local_path = f"/content/{filename}.tif"
        urllib.request.urlretrieve(download_url, local_path)

        # 4. บังคับป็อปอัปดาวน์โหลดลงคอมพิวเตอร์
        print(f"📦 เริ่มการดาวน์โหลดไฟล์ {filename}.tif ลงเครื่องคอมพิวเตอร์ของคุณแล้ว...")
        files.download(local_path)

        # 5. หน่วง 3 วินาที กันคิวดาวน์โหลดชนกัน
        time.sleep(3)
    except Exception as e:
        print(f"❌ เกิดข้อผิดพลาดในการดาวน์โหลด {filename}: {e}")

# สั่งส่งออก NDWI Float หน้าแล้ง (เม.ย.-พ.ค.) ทั้ง 3 ปี
direct_download_ndwi_tiff(ndwi_1993, 'NDWI_Float_Dry_1993_L5')
direct_download_ndwi_tiff(ndwi_2003, 'NDWI_Float_Dry_2003_L7')
direct_download_ndwi_tiff(ndwi_2026, 'NDWI_Float_Dry_2026_L9')

print("\n✨ ดำเนินการส่งข้อมูลดาวน์โหลดเรียบร้อยแล้ว!")
print("💡 เปิดใน QGIS: Symbology → Singleband pseudocolor → กำหนด min = -1, max = 1")